In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
HERE = %pwd
sys.path.append(os.path.dirname(HERE))

%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display
    
import numpy as np
import pandas as pd
import copy
import pickle
import time
import collections
from tqdm import tqdm
from collections import defaultdict

In [2]:
from src import utils
rng = utils.set_seed()

dir_parent = utils.dir_parent
version_exp = utils.version_exp
dir_workspace = f"{dir_parent}/research/TFCSR"

device_emb = utils.device
device_reranker = utils.device

In [3]:
data_name = "Job"
N_icl = [1]
flag_replace_NER = False

from src.data_loader import Loader
loader = Loader(dir_workspace, version_exp, data_name, N_icl=N_icl, flag_replace_NER=flag_replace_NER)    
dict_data = loader.load_data()

# flag
dict_flag = dict_data["flag"]

# queries
dict_text = dict_data["profile"]
for text_type in ["concat"]:
    dict_text.update(dict_data[text_type])

# all candidate items
d_documents = dict_data["items"]["candidates"]

user_type = 'profile_mid-career'

users = list(dict_flag[user_type].keys())


d_sample = dict()
for user in users:
    try:
        user_text = dict_text[f"concat_{user_type}"][user]
    except:
        user_text = dict_text[user_type][user]
    
    s_flag = pd.Series(dict_flag[user_type][user])  
    df_candidates = pd.DataFrame({
        "item_text" : pd.Series({item : d_documents[item] for item in s_flag.index}),
        "flag" : s_flag
    })

    d_sample[user] = {
        "query" : user_text,
        "candidates" : df_candidates["item_text"]
    }

In [4]:
d_documents = pd.concat([d["candidates"] for d in d_sample.values()]).drop_duplicates().to_dict()
len(d_documents)

1891

In [5]:
s = pd.Series({user : utils.compute_token(d["query"]) for user, d in d_sample.items()})
print(f"query {s.mean():.1f} \\pm {s.std():.1f}")

query 105.2 \pm 18.2


In [6]:
s = pd.Series({item : utils.compute_token(item_text) for item, item_text in d_documents.items()})
print(f"item {s.mean():.1f} \\pm {s.std():.1f}")

item 333.2 \pm 207.6


In [7]:
def _time(tmp_fn):
    start_time = time.time()
    tmp_fn(dummy=None)
    elapsed_time = time.time() - start_time
    return elapsed_time

dd_stat = dict()

In [8]:
path_ = "./cost_emb_qwen3-8b.pickle"
try:
    with open(path_, 'rb') as f:
        d_stat = pickle.load(f)  
except:
    d_stat = dict()
    
    # load embedding model
    model_name_emb = "Qwen/Qwen3-Embedding-8B"
    from src.embedding import Embedding
    model_id = f"{dir_parent}/models/embedding_models/{model_name_emb}"
    emb = Embedding(model_id, device_emb)
    
    # user embedding
    fn = lambda t : emb.encode(t, query=True)
    _time(lambda dummy : fn("test query"))
    
    s = pd.Series({user : _time(lambda dummy : fn(d["query"])) for user, d in tqdm(d_sample.items())})
    print(f"{model_name_emb} query: {s.mean():.3f} \\pm {s.std():.3f}, {s.sum():.1f}")
    d_stat[f"{model_name_emb} query"] = s
    
    # item embedding (user)
    fn = lambda d_ : {
        item : emb.encode(item_text, query=False)
        for item, item_text in d_.to_dict().items()
    }
    s = pd.Series({user : _time(lambda dummy : fn(d["candidates"])) for user, d in tqdm(d_sample.items())})
    print(f"{model_name_emb} candidates(user): {s.mean():.3f} \\pm {s.std():.3f}, {s.sum():.1f}")
    d_stat[f"{model_name_emb} candidates(user)"] = s
    
    # item embedding (pre-computed)
    s = pd.Series({item : _time(lambda dummy : emb.encode(t, query=False)) for item, t in tqdm(d_documents.items())})
    print(f"{model_name_emb} candidates(pre): {s.mean():.3f} \\pm {s.std():.3f}, {s.sum():.1f}")
    d_stat[f"{model_name_emb} candidates(pre)"] = s

    with open(path_, 'wb') as f:
        pickle.dump(d_stat, f)  
        
dd_stat.update(d_stat)

In [9]:
path_ = "./cost_emb_nv-8b.pickle"
try:
    with open(path_, 'rb') as f:
        d_stat = pickle.load(f)  
except:
    d_stat = dict()
    
    # load embedding model
    model_name_emb = "nvidia/llama-embed-nemotron-8b"
    from src.embedding import Embedding
    model_id = f"{dir_parent}/models/embedding_models/{model_name_emb}"
    emb = Embedding(model_id, device_emb)
    
    # user embedding
    fn = lambda t : emb.encode(t, query=True)
    _time(lambda dummy : fn("test query"))
    
    s = pd.Series({user : _time(lambda dummy : fn(d["query"])) for user, d in tqdm(d_sample.items())})
    print(f"{model_name_emb} query: {s.mean():.3f} \\pm {s.std():.3f}, {s.sum():.1f}")
    d_stat[f"{model_name_emb} query"] = s
    
    # item embedding (user)
    fn = lambda d_ : {
        item : emb.encode(item_text, query=False)
        for item, item_text in d_.to_dict().items()
    }
    s = pd.Series({user : _time(lambda dummy : fn(d["candidates"])) for user, d in tqdm(d_sample.items())})
    print(f"{model_name_emb} candidates(user): {s.mean():.3f} \\pm {s.std():.3f}, {s.sum():.1f}")
    d_stat[f"{model_name_emb} candidates(user)"] = s
    
    # item embedding (pre-computed)
    s = pd.Series({item : _time(lambda dummy : emb.encode(t, query=False)) for item, t in tqdm(d_documents.items())})
    print(f"{model_name_emb} candidates(pre): {s.mean():.3f} \\pm {s.std():.3f}, {s.sum():.1f}")
    d_stat[f"{model_name_emb} candidates(pre)"] = s

    with open(path_, 'wb') as f:
        pickle.dump(d_stat, f)  

dd_stat.update(d_stat)

In [10]:
path_ = "./cost_reranker_qwen3-8b.pickle"
try:
    with open(path_, 'rb') as f:
        d_stat = pickle.load(f)  
except:
    d_stat = dict()
    
    model_name_reranker = "Qwen/Qwen3-Reranker-8B"
    from src.reranker import Reranker
    model_id = f"{dir_parent}/models/reranker_models/{model_name_reranker}"
    reranker = Reranker(model_id, device_reranker)
    
    # user-item pair
    fn = lambda d : reranker.compute([d["query"]], d["candidates"].values)
    s = pd.Series({user : _time(lambda dummy : fn(d)) for user, d in tqdm(d_sample.items())})
    print(f"{model_name_reranker} {s.mean():.3f} \\pm {s.std():.3f}, {s.sum():.1f}")
    d_stat[f"{model_name_reranker}"] = s
    
    with open(path_, 'wb') as f:
        pickle.dump(d_stat, f)  

dd_stat.update(d_stat)

In [11]:
# LLM: load time and cost from raw reranking data
dir_llm = f"{dir_workspace}/LLMreranking_data/{version_exp}/{data_name}/candidate50_atK10"

def _compute_api_cost(model_name, total_input_tokens, total_output_tokens):
    """Replicate LLM.compute_cost without instantiating the full LLM class."""
    s = pd.Series([total_input_tokens, total_output_tokens])
    if "5.4" in model_name:
        return (s.iloc[0] * 2.5 + s.iloc[1] * 15) / 1e6
    elif "5.1" in model_name:
        return (s.iloc[0] * 1.25 + s.iloc[1] * 10) / 1e6
    elif "sonnet-4" in model_name:
        return (s.iloc[0] * 3 + s.iloc[1] * 15) / 1e6
    else:
        raise ValueError(f"Unknown model: {model_name}")

import glob

llm_models = {
    "gpt-5.1-2025-11-13_reasoning_none": "gpt-5_1-2025-11-13_reasoning_none",
    "gpt-5.4-2026-03-05_reasoning_none": "gpt-5_4-2026-03-05_reasoning_none",
    "us.anthropic.claude-sonnet-4-5-20250929-v1:0": "us_anthropic_claude-sonnet-4-5-20250929-v1_0",
}

text_type = f"{user_type}"  # profile_mid-career

for model_name, model_dir_name in llm_models.items():
    dir_res = f"{dir_llm}/{text_type}_{model_dir_name}_instdefault_original"
    files = sorted(glob.glob(f"{dir_res}/*.pickle"))
    assert len(files) > 0, f"No files found in {dir_res}"

    ddict_res = {}
    for path in files:
        with open(path, 'rb') as f:
            ddict_res.update(pickle.load(f))

    # per-user time and tokens
    d_time, d_in, d_out = {}, {}, {}
    for user, d in ddict_res.items():
        log_entries = d["log"]
        d_time[user] = sum(v["time"] for v in log_entries.values())
        d_in[user]   = sum(v["input token"] for v in log_entries.values())
        d_out[user]  = sum(v["output token"] for v in log_entries.values())

    s_time = pd.Series(d_time)
    s_in   = pd.Series(d_in)
    s_out  = pd.Series(d_out)

    # per-user cost
    # Use per-user token counts -> per-user cost
    s_cost = pd.Series({
        user: _compute_api_cost(model_name, d_in[user], d_out[user])
        for user in d_time
    })

    dd_stat[f"{model_name} time"] = s_time
    dd_stat[f"{model_name} cost"] = s_cost

    total_cost = _compute_api_cost(model_name, s_in.sum(), s_out.sum())
    print(f"{model_name}: {len(s_time)} users, "
          f"total_time={s_time.sum():.1f}s, total_cost=${total_cost:.4f}")

gpt-5.1-2025-11-13_reasoning_none: 500 users, total_time=1108.6s, total_cost=$9.8297
gpt-5.4-2026-03-05_reasoning_none: 500 users, total_time=972.6s, total_cost=$19.5144
us.anthropic.claude-sonnet-4-5-20250929-v1:0: 500 users, total_time=3538.3s, total_cost=$28.9082


In [12]:
s = pd.DataFrame(dd_stat).sum()
s

Qwen/Qwen3-Embedding-8B query                          20.624064
Qwen/Qwen3-Embedding-8B candidates(user)             1220.316137
Qwen/Qwen3-Embedding-8B candidates(pre)               100.439001
nvidia/llama-embed-nemotron-8b query                   18.032044
nvidia/llama-embed-nemotron-8b candidates(user)      1106.388766
nvidia/llama-embed-nemotron-8b candidates(pre)         92.072411
Qwen/Qwen3-Reranker-8B                               4140.286020
gpt-5.1-2025-11-13_reasoning_none time               1108.621067
gpt-5.1-2025-11-13_reasoning_none cost                  9.829695
gpt-5.4-2026-03-05_reasoning_none time                972.613144
gpt-5.4-2026-03-05_reasoning_none cost                 19.514390
us.anthropic.claude-sonnet-4-5-20250929-v1:0 time    3538.309157
us.anthropic.claude-sonnet-4-5-20250929-v1:0 cost      28.908246
dtype: float64

In [13]:
s = pd.DataFrame(dd_stat).sum()

s_local_time = s.loc[[i for i in s.index if "Qwen" in i]]

s_ = s.loc[[i for i in s.index if "time" in i]]
s_.index = [i.split(" ")[0] for i in s_.index]
s_llm_time = s_.copy()

s_ = s.loc[[i for i in s.index if "cost" in i]]
s_.index = [i.split(" ")[0] for i in s_.index]
s_llm_cost = s_.copy()

In [14]:
s_time = pd.concat([s_local_time, s_llm_time], axis=0)

s_local_cost = s_local_time * 1.861 / 3600
s_cost = pd.concat([s_local_cost, s_llm_cost], axis=0)
df = pd.concat([s_time, s_cost], axis=1)
df.columns = ["time", "cost"]

def _tmp(i):
    d_ = {
        "Qwen/Qwen3-Reranker-0.6B" : "Qwen3-R (0.6B)",
        "Qwen/Qwen3-Embedding-0.6B" : "Qwen3-E (0.6B)",
        "Qwen/Qwen3-Reranker-8B" : "Qwen3-R (8B)",
        "Qwen/Qwen3-Embedding-8B" : "Qwen3-E (8B)",
        "gpt-5.1-2025-11-13_reasoning_none": "GPT-5.1",
        "gpt-5.4-2026-03-05_reasoning_none": "GPT-5.4",
        "us.anthropic.claude-sonnet-4-5-20250929-v1:0": "Sonnet 4.5",
    }
    for k,v in d_.items():
        i = i.replace(k,v)
    return i

df.index = [_tmp(i) for i in df.index]
display(df.round(4))

df_ = df.copy()
df_["time"] = df_["time"].map(lambda s : f"${s:.1f}$")
df_["cost"] = df_["cost"].map(lambda s : f"(${s:.3f}$)")
print(df_.to_latex(escape=False))

,time,cost
Qwen3-E (8B) query,20.6241,0.0107
Qwen3-E (8B) candidates(user),1220.3161,0.6308
Qwen3-E (8B) candidates(pre),100.4390,0.0519
Qwen3-R (8B),4140.2860,2.1403
GPT-5.1,1108.6211,9.8297
GPT-5.4,972.6131,19.5144
Sonnet 4.5,3538.3092,28.9082


\begin{tabular}{lll}
\toprule
 & time & cost \\
\midrule
Qwen3-E (8B) query & $20.6$ & ($0.011$) \\
Qwen3-E (8B) candidates(user) & $1220.3$ & ($0.631$) \\
Qwen3-E (8B) candidates(pre) & $100.4$ & ($0.052$) \\
Qwen3-R (8B) & $4140.3$ & ($2.140$) \\
GPT-5.1 & $1108.6$ & ($9.830$) \\
GPT-5.4 & $972.6$ & ($19.514$) \\
Sonnet 4.5 & $3538.3$ & ($28.908$) \\
\bottomrule
\end{tabular}

